In [ ]:
"""
CV grid for Persistence Landscapes (H0+H1)

Output:
  - grid_PL_H0H1.csv
"""

import os
import numpy as np
import pandas as pd

from gudhi.representations import Landscape
from scipy.stats import mannwhitneyu, combine_pvalues
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import RepeatedStratifiedKFold

# Configuration

BASE = "RIPS"

RESOLUTIONS     = [100, 125, 150]
N_LANDSCAPES    = [1, 3, 5]
NORMALIZATIONS  = ["none", "l1"]
SIGMAS          = [0, 1, 2]
THR_OPTIONS     = ["0", "p10"]

EPS = 1e-12

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 0

OUTDIR = "PL_MWU"
os.makedirs(OUTDIR, exist_ok=True)

# LOADER
def read_and_save(filedir, tube):
    if tube and tube[0] != ".":
        _, ext = os.path.splitext(os.path.join(filedir, tube))
        tubenamerips = tube.split("_")[-1].split(".")[0]
        if ext != ".pdf" and tubenamerips == "Rips0":
            r0_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips0.txt")
            r1_path = os.path.join(filedir, "_".join(tube.split("_")[:-1]) + "_Rips1.txt")

            Rips0 = np.array(pd.read_csv(r0_path, sep=" ", header=None))
            if len(Rips0) and np.isinf(Rips0[-1, 1]):
                Rips0 = Rips0[:-1]

            Rips1 = np.array(pd.read_csv(r1_path, sep=" ", header=None))
            if len(Rips1) and np.isnan(Rips1[-1, 1]):
                Rips1[-1, 1] = 0

            return [[Rips0, Rips1], None, None]
    return []

def list_patients(root_dir):
    return [x for x in sorted(os.listdir(root_dir)) if not x.startswith(".")]

def find_rips0_file(patient_dir):
    for f in sorted(os.listdir(patient_dir)):
        if f.startswith("."):
            continue
        if f.endswith("_Rips0.txt"):
            return f
    return None

def load_group_diagrams(group_root):
    out = {}
    for patient in list_patients(group_root):
        p_dir = os.path.join(group_root, patient)
        rips0_file = find_rips0_file(p_dir)
        if rips0_file is None:
            continue
        data = read_and_save(p_dir, rips0_file)
        if not data:
            continue
        diagrams = data[0]
        if diagrams is None or len(diagrams) < 2:
            continue
        out[patient] = diagrams
    return out


def collect_persistences_from_samples(diagrams_list, dim):
    pers = []
    for diagrams in diagrams_list:
        pairs = diagrams[dim]
        pairs = np.asarray(pairs, dtype=float)
        p = pairs[:, 1] - pairs[:, 0]
        p = p[np.isfinite(p)]
        p = p[p > 0]
        if p.size:
            pers.append(p)
    return np.concatenate(pers) if pers else np.array([])

def apply_persistence_threshold(pairs, thr):
    pairs = np.asarray(pairs, dtype=float)
    pers = pairs[:, 1] - pairs[:, 0]
    keep = np.isfinite(pers) & (pers >= thr)
    return pairs[keep]

def landscape_from_pairs(pairs, resolution, n_landscapes):
    if pairs is None or len(pairs) == 0:
        return np.zeros(n_landscapes * resolution)
    return Landscape(num_landscapes=n_landscapes, resolution=resolution).fit_transform([pairs])[0]

def normalize_curve(curve, mode):
    if mode == "l1":
        return curve / (np.sum(np.abs(curve)) + EPS)
    return curve

def smooth_landscape(curve, sigma, resolution, n_landscapes):
    if sigma <= 0:
        return curve
    v = curve.reshape(n_landscapes, resolution)
    v = gaussian_filter1d(v, sigma=sigma, axis=1)
    return v.reshape(-1)

def mannwhitney_per_bin(X_nr, X_r):
    pvals = []
    for j in range(X_nr.shape[1]):
        _, p = mannwhitneyu(X_nr[:, j], X_r[:, j])
        pvals.append(p)
    return np.array(pvals)

def fisher_combine(pvals):
    return combine_pvalues(np.clip(pvals, EPS, 1))[1]

# Main
if __name__ == "__main__":

    NR = load_group_diagrams(os.path.join(BASE, "NonRelapse"))
    R  = load_group_diagrams(os.path.join(BASE, "Relapse"))

    X = list(NR.values()) + list(R.values())
    y = np.array([0]*len(NR) + [1]*len(R))

    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

    rows = []

    for sigma in SIGMAS:
        for thr_opt in THR_OPTIONS:
            for res in RESOLUTIONS:
                for n_land in N_LANDSCAPES:
                    for norm in NORMALIZATIONS:

                        scores = []

                        for tr, te in cv.split(np.zeros(len(y)), y):

                            X_tr = [X[i] for i in tr]
                            X_te = [X[i] for i in te]
                            y_te = y[te]

                            if thr_opt == "0":
                                thr0 = 0
                                thr1 = 0
                            else:
                                thr0 = np.percentile(collect_persistences_from_samples(X_tr,0),10)
                                thr1 = np.percentile(collect_persistences_from_samples(X_tr,1),10)

                            X0, X1 = [], []

                            for d, lab in zip(X_te, y_te):

                                c0 = smooth_landscape(
                                    normalize_curve(
                                        landscape_from_pairs(
                                            apply_persistence_threshold(d[0], thr0),
                                            res, n_land
                                        ), norm),
                                    sigma, res, n_land
                                )

                                c1 = smooth_landscape(
                                    normalize_curve(
                                        landscape_from_pairs(
                                            apply_persistence_threshold(d[1], thr1),
                                            res, n_land
                                        ), norm),
                                    sigma, res, n_land
                                )

                                feat = np.concatenate([c0,c1])

                                if lab == 0: X0.append(feat)
                                else: X1.append(feat)

                            if len(X0)<2 or len(X1)<2:
                                score = 0
                            else:
                                X0 = np.vstack(X0)
                                X1 = np.vstack(X1)

                                score = -np.log10(max(fisher_combine(
                                    mannwhitney_per_bin(X0,X1)
                                ), EPS))

                            scores.append(score)

                        rows.append({
                            "dimension":"0+1",
                            "sigma":sigma,
                            "thr_opt":thr_opt,
                            "resolution":res,
                            "n_landscapes":n_land,
                            "normalization":norm,
                            "score_median_CV":np.median(scores),
                            "score_mean_CV":np.mean(scores)
                        })

    df = pd.DataFrame(rows).sort_values("score_median_CV",ascending=False)

    out = os.path.join(OUTDIR,"grid_PL_H0H1.csv")
    df.to_csv(out,index=False)

    print("Saved:", out)